# Comparing Docker Networking Drivers

This notebook walks through the four main Docker networking drivers — **bridge**, **host**, **overlay**, and **macvlan** — with practical examples showing when each one makes sense.

## Why this matters

Docker networking determines how containers talk to each other and to the outside world. Choosing the wrong driver can lead to isolation bugs, performance hits, or port-conflict headaches. This notebook is a quick reference I can come back to when I need to pick a driver for a new project.

## 1. Bridge networking (default)

Bridge is the default Docker network. Containers on a bridge network get their own IP on a private Docker-internal subnet, and Docker handles NAT to expose ports to the host.

**When to use it:** Single-host deployments, dev/test environments, anything where you want container isolation with port publishing.

**When to avoid it:** Multi-host communication (overlay is better), or when you need containers to appear as real machines on the LAN (macvlan).

In [ ]:
# Create a custom bridge network
!docker network create --driver bridge my-bridge-net

# List networks to confirm
!docker network ls | grep my-bridge-net

In [ ]:
# Run two containers on the bridge network and ping from one to the other
!docker run -d --name bridge-host1 --network my-bridge-net nginx:alpine
!docker run -d --name bridge-host2 --network my-bridge-net nginx:alpine

# Get the IP of bridge-host2 and ping it from bridge-host1
!HOST2_IP=$(docker inspect -f '{{range .NetworkSettings.Networks}}{{.IPAddress}}{{end}}' bridge-host2) && echo "bridge-host2 IP: $HOST2_IP" && docker exec bridge-host1 ping -c 2 $HOST2_IP

In [ ]:
# Clean up
!docker rm -f bridge-host1 bridge-host2
!docker network rm my-bridge-net

## 2. Host networking

The host driver removes network isolation entirely — the container shares the host's network namespace directly. There's no NAT, no port publishing, and the container binds to the host's interfaces.

**When to use it:** Performance-critical workloads (high-throughput proxies, monitoring agents), or when you need the container to see all host network interfaces.

**When to avoid it:** Any situation where you need port isolation between containers or between container and host. Also doesn't work on Docker Desktop for Mac/Windows — Linux only.

In [ ]:
# Run an nginx container with host networking
# It binds directly to the host's port 8080
!docker run -d --name host-demo --network host nginx:alpine

# Verify it's listening on the host's port 80
!curl -s -o /dev/null -w '%{http_code}' http://localhost:80

In [ ]:
# Clean up
!docker rm -f host-demo

## 3. Overlay networking

Overlay networks span multiple Docker hosts. They use VXLAN tunneling to create a flat network across a swarm or a standalone Docker setup with key-value store discovery.

**When to use it:** Multi-host container communication, Docker Swarm services, or any setup where containers on different machines need to reach each other by name.

**When to avoid it:** Single-host setups (bridge is simpler and has less overhead). Requires a key-value store or swarm mode for standalone Docker.

In [ ]:
# Initialize a Docker swarm (overlay requires swarm or a KV store)
!docker swarm init 2>/dev/null || echo 'Swarm already active'

# Create an overlay network
!docker network create --driver overlay --attachable my-overlay-net

# List and inspect the overlay network
!docker network inspect my-overlay-net --format '{{.Driver}} {{.Scope}}'

In [ ]:
# Run a service on the overlay network
# On a single host this still demonstrates the overlay driver
!docker service create --name overlay-web --network my-overlay-net --replicas 1 -p 8081:80 nginx:alpine

# Verify the service is running
!docker service ls | grep overlay-web

In [ ]:
# Clean up
!docker service rm overlay-web
!docker network rm my-overlay-net
!docker swarm leave --force 2>/dev/null || true

## 4. Macvlan networking

Macvlan assigns a real MAC address to each container, making it appear as a physical device on the network. The container gets an IP from the same subnet as the host.

**When to use it:** Legacy applications that expect to be directly on the LAN, or when you need containers to be reachable by their own IP without port mapping.

**When to avoid it:** Cloud environments where you don't control the underlying network (AWS, GCP typically block promiscuous mode). Also requires careful IP planning since each container consumes a real IP.

In [ ]:
# Create a macvlan network
# Adjust the subnet and gateway to match your actual LAN
# This example uses a /24 on the host's eth0 subnet
!docker network create --driver macvlan --subnet=192.168.1.0/24 --gateway=192.168.1.1 -o parent=eth0 my-macvlan-net 2>/dev/null || echo 'Network creation skipped (may need real LAN details)'

In [ ]:
# If the macvlan network was created, run a container on it
# The container will appear as a distinct device on the LAN
# docker run -d --name macvlan-demo --network my-macvlan-net --ip 192.168.1.100 nginx:alpine
# docker exec macvlan-demo ip addr show

print('Macvlan demo skipped — requires a real LAN subnet to run.')

In [ ]:
# Clean up
!docker rm -f macvlan-demo 2>/dev/null || true
!docker network rm my-macvlan-net 2>/dev/null || true

## Quick comparison table

| Driver | Scope | Isolation | Port publishing | Multi-host | Performance |
|--------|-------|-----------|-----------------|------------|-------------|
| bridge | host | Container → host NAT | Yes | No | Good |
| host | host | None (shared namespace) | No (binds directly) | No | Best |
| overlay | swarm | Full | Yes | Yes (VXLAN) | Good |
| macvlan | host | Container = real LAN device | No (direct IP) | No | Best (LAN) |

## Decision cheat sheet

- **Single host, simple setup** → bridge (default, just works)
- **High performance, no isolation needed** → host
- **Multi-host or swarm** → overlay
- **Legacy app that needs a real LAN IP** → macvlan

For most dev/test work, bridge is the right call. Reach for host when profiling or running agents. Overlay is the go-to for distributed deployments. Macvlan is niche but powerful when you need it.

## What I learned

- Bridge is the safe default — it just works and gives you port isolation via NAT.
- Host networking is fast but you lose all isolation; two containers can't bind the same port.
- Overlay adds overhead (VXLAN encapsulation) but is the only option for multi-host without macvlan.
- Macvlan is powerful for on-prem legacy apps but doesn't play well with most cloud providers.

Next step: try wiring an overlay network into a Swarm stack and compare latency against bridge.